# RK1828 Qwen3-Embedding Text Vector Demo


[Qwen3-Embedding-0.6B](https://github.com/QwenLM/Qwen3-Embedding): encodes
text into **1024-dim vectors** for semantic search / RAG retrieval. GRQ W4A16
quantized, measured 76.7 ms prefill per sentence; paraphrase cosine similarity
0.958 vs 0.283 for an unrelated text.

| Step | Content |
|---|---|
| 1 | Download models (from huggingface.co) |
| 2 | Extract the prebuilt demo |
| 3 | Enter sentences and batch-encode |
| 4 | Cosine similarity matrix |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("qwen3-embedding-0.6b-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/qwen3_embedding/demo_x86_64.tar.gz"),
    ("qwen3-embedding-0.6b-rknn3-rk1828", "model/Qwen3-Embedding-0.6B.embed.bin", "demos/qwen3_embedding/model/Qwen3-Embedding-0.6B.embed.bin"),
    ("qwen3-embedding-0.6b-rknn3-rk1828", "model/Qwen3-Embedding-0.6B.rknn", "demos/qwen3_embedding/model/Qwen3-Embedding-0.6B.rknn"),
    ("qwen3-embedding-0.6b-rknn3-rk1828", "model/Qwen3-Embedding-0.6B.tokenizer.gguf", "demos/qwen3_embedding/model/Qwen3-Embedding-0.6B.tokenizer.gguf"),
    ("qwen3-embedding-0.6b-rknn3-rk1828", "model/Qwen3-Embedding-0.6B.weight", "demos/qwen3_embedding/model/Qwen3-Embedding-0.6B.weight"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Extract the prebuilt demo

`demo_x86_64.tar.gz` contains the official x86_64 C++ demo and its runtime
libraries (`lib/`). Hugging Face does not host raw executables, so they are
shipped as a tarball; extract once (re-running skips automatically).

> **Why a C++ demo instead of pure Python?**
> Notebooks 01-05 (classification / detection / FastSAM / pose / segmentation)
> are a single model with one forward pass and simple post-processing, so they
> walk through everything in Python (`rknn3.py`). This demo's pipeline is far
> more complex — chained models, tokenizer, KV cache, sliding-window /
> streaming state, hundreds of lines of orchestration — reimplementing it in
> Python would be costly and risk diverging from the official behavior.
> So we run the official `rknn3-model-zoo` C++ demo directly; source code:
> [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> under `examples/`.

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/qwen3_embedding/demo_x86_64.tar.gz", "demos/qwen3_embedding")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. Enter sentences and encode

One sentence per line. The demo encodes one sentence per run and writes the
1024-dim float32 vector to `embedding_out.bin`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_texts = widgets.Textarea(
    value="How do I reset my password?\n"
          "I forgot my password, how can I change it?\n"
          "What's the best way to cook rice?\n"
          "如何重置我的密码？",
    description="Sentences (one per line):", layout=widgets.Layout(width="75%", height="120px"))
display(w_texts)
print("Then keep running downwards ▼")

In [ ]:
import numpy as np

DEMO = "demos/qwen3_embedding"
texts = [t.strip() for t in w_texts.value.splitlines() if t.strip()]
embs = []
for t in texts:
    run_demo(["./rknn_qwen3_embedding_demo",
              "model/Qwen3-Embedding-0.6B.rknn", "model/Qwen3-Embedding-0.6B.weight",
              "model/Qwen3-Embedding-0.6B.tokenizer.gguf",
              "model/Qwen3-Embedding-0.6B.embed.bin",
              "0xff", t], DEMO)
    embs.append(np.fromfile(f"{DEMO}/embedding_out.bin", dtype=np.float32))
embs = np.stack(embs)
print("embedding matrix:", embs.shape)

## 4. Cosine similarity matrix

Semantically close sentences (even across languages) should score near 1,
unrelated ones near 0.

In [ ]:
import matplotlib.pyplot as plt
from vision_post import setup_cjk_font

setup_cjk_font()
norm = embs / np.linalg.norm(embs, axis=1, keepdims=True)
sim = norm @ norm.T

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(texts)), [t[:18] for t in texts], rotation=45, ha="right")
ax.set_yticks(range(len(texts)), [t[:18] for t in texts])
for i in range(len(texts)):
    for j in range(len(texts)):
        ax.text(j, i, f"{sim[i,j]:.2f}", ha="center", va="center",
                color="white" if sim[i, j] > 0.6 else "black")
plt.colorbar(im)
plt.title("cosine similarity")
plt.tight_layout()
plt.show()

## Summary

- An embedding model only measures "semantic distance", regardless of
  language — try mixed Chinese/English paraphrases.
- This is the retrieval side of a RAG system; pair it with the Reranker in
  notebook 17 for the full retrieve + rerank pipeline.